# Klasifikasi Kepuasan Pelanggan Produk GEOFFMAX
## Berdasarkan Ulasan E-Commerce Shopee — Algoritma Naïve Bayes (CRISP-DM)

| | |
|---|---|
| **Algoritma** | Multinomial Naïve Bayes |
| **Metodologi** | CRISP-DM |
| **Validasi** | 10-Fold Stratified Cross Validation |
| **Kelas** | Tidak Puas · Cukup Puas · Puas |
| **Data** | 400 ulasan produk sepatu GEOFFMAX di Shopee |
| **Kolom** | `ulasan` (teks) · `rating` (1–5) |


## Sel 0 — Install & Import Library

In [ ]:
!pip install PySastrawi -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import joblib
import warnings
warnings.filterwarnings('ignore')

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (confusion_matrix, classification_report,
                              accuracy_score, roc_auc_score, roc_curve)
from sklearn.preprocessing import label_binarize

np.random.seed(42)
print("✅ Semua library berhasil diimport.")


## Sel 1 — Upload Data Asli GEOFFMAX
> Upload file `data_ulasan_dublin_bersih__1_.csv` saat tombol muncul.


In [ ]:
from google.colab import files
uploaded = files.upload()
nama_file = list(uploaded.keys())[0]

df_raw = pd.read_csv(nama_file)
print(f"✅ File berhasil dimuat: {nama_file}")
print(f"   Jumlah baris  : {len(df_raw)}")
print(f"   Kolom         : {list(df_raw.columns)}")
df_raw.head()


## Sel 1b — Tambah Data Baru (Khusus Ulasan Rating 3)
> Untuk mengatasi imbalance kelas **Cukup Puas** (hanya 80 data vs 136/184 di kelas lain).
> Kumpulkan ulasan GEOFFMAX rating 3 tambahan (kolom `ulasan`, `rating`), simpan sebagai CSV,
> lalu upload di sel ini. Kalau belum ada data tambahan, LEWATI sel ini (lanjut ke Sel 2).

In [ ]:
# Upload file tambahan (format kolom harus SAMA: ulasan, rating)
uploaded_tambahan = files.upload()

if len(uploaded_tambahan) > 0:
    nama_file_tambahan = list(uploaded_tambahan.keys())[0]
    df_tambahan = pd.read_csv(nama_file_tambahan)

    print(f"📥 Data tambahan dimuat: {nama_file_tambahan}")
    print(f"   Jumlah baris tambahan : {len(df_tambahan)}")
    print(f"   Distribusi rating tambahan:")
    print(df_tambahan['rating'].value_counts().sort_index().to_string())

    jumlah_sebelum = len(df_raw)

    # Gabungkan dengan data asli
    df_raw = pd.concat([df_raw, df_tambahan], ignore_index=True)

    # Buang duplikat ulasan persis sama (kalau ada yang ke-scrape 2x)
    jumlah_sebelum_dedup = len(df_raw)
    df_raw = df_raw.drop_duplicates(subset='ulasan').reset_index(drop=True)
    jumlah_duplikat = jumlah_sebelum_dedup - len(df_raw)

    print(f"\n✅ Data digabung: {jumlah_sebelum} (lama) + {len(df_tambahan)} (baru) "
          f"- {jumlah_duplikat} (duplikat) = {len(df_raw)} total")
    print(f"\nDistribusi rating SETELAH digabung:")
    print(df_raw['rating'].value_counts().sort_index().to_string())
else:
    print("⏭️ Tidak ada file diupload — lanjut pakai data asli tanpa tambahan.")


## Sel 2 — Data Understanding (CRISP-DM Tahap 2)

In [ ]:
# Buat label Y dari rating
def buat_label(r):
    if r <= 2:   return 0, 'Tidak Puas'
    elif r == 3: return 1, 'Cukup Puas'
    else:        return 2, 'Puas'

df_raw[['Y_Kode','Y_Label']] = df_raw['rating'].apply(
    lambda r: pd.Series(buat_label(r)))

# Statistik dasar
print("=== INFO DATASET ===")
print(f"Total ulasan   : {len(df_raw)}")
print(f"Missing values : {df_raw.isnull().sum().sum()}")
print()
print("=== DISTRIBUSI RATING ===")
print(df_raw['rating'].value_counts().sort_index().to_string())
print()
print("=== DISTRIBUSI KELAS Y (3 kelas) ===")
dist = df_raw['Y_Label'].value_counts()
pct  = df_raw['Y_Label'].value_counts(normalize=True)*100
print(pd.DataFrame({'Jumlah': dist, 'Persentase (%)': pct.round(2)}))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Grafik distribusi rating
rating_cnt = df_raw['rating'].value_counts().sort_index()
bars = axes[0].bar(rating_cnt.index, rating_cnt.values,
                   color=['#E74C3C','#E74C3C','#F39C12','#27AE60','#27AE60'],
                   edgecolor='white', width=0.6)
for bar, v in zip(bars, rating_cnt.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+0.5,
                 str(v), ha='center', fontweight='bold', fontsize=11)
axes[0].set_title('Distribusi Rating Bintang', fontsize=13)
axes[0].set_xlabel('Rating'); axes[0].set_ylabel('Jumlah Ulasan')
axes[0].set_xticks([1,2,3,4,5])

# Grafik distribusi kelas Y
order  = ['Tidak Puas', 'Cukup Puas', 'Puas']
warna  = ['#E74C3C', '#F39C12', '#27AE60']
counts = [df_raw[df_raw['Y_Label']==l].shape[0] for l in order]
bars2  = axes[1].bar(order, counts, color=warna, edgecolor='white', width=0.5)
for bar, v in zip(bars2, counts):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+0.5,
                 str(v), ha='center', fontweight='bold', fontsize=11)
axes[1].set_title('Distribusi Kelas Kepuasan (Y)', fontsize=13)
axes[1].set_ylabel('Jumlah Ulasan')

plt.tight_layout()
plt.savefig('distribusi_kelas.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik tersimpan sebagai distribusi_kelas.png")


## Sel 3 — Data Preparation (CRISP-DM Tahap 3)

Tahapan preprocessing teks:
1. **Case folding** — semua huruf kecil
2. **Cleaning** — hapus angka, tanda baca, karakter khusus
3. **Stopword removal** — Bahasa Indonesia (Sastrawi) + slang
4. **Stemming** — ubah kata ke bentuk dasar
5. **TF-IDF Vectorization** — ubah teks → vektor numerik


In [ ]:
# Inisialisasi Sastrawi
factory_sw = StopWordRemoverFactory()
stopwords  = set(factory_sw.get_stop_words())

# Tambah stopwords slang e-commerce
slang_sw = {'yg','dgn','utk','krn','sdh','blm','ga','gak','nggak',
            'udah','udh','bgt','aja','sih','deh','nih','loh','dong',
            'emg','emang','tp','sy','gw','lo','lu','wkwk','haha',
            'lah','kah','nya','si','ok','oke','ya','yah'}

# PENTING: kata negasi/kontras JANGAN ikut dihapus meski ada di daftar stopword Sastrawi/slang.
# Kata-kata ini justru penanda utama ulasan bersentimen CAMPURAN (kelas 'Cukup Puas'),
# contoh: 'bagus TAPI kurang nyaman', 'nyaman TAPI lem meleber'.
# Kalau dihapus, sinyal kontrasnya hilang dan 'Cukup Puas' jadi mirip 'Puas'/'Tidak Puas' murni.
kata_penting_sentimen = {'tidak','belum','tapi','tetapi','namun','tp'}

all_stopwords = (stopwords | slang_sw) - kata_penting_sentimen

factory_st = StemmerFactory()
stemmer    = factory_st.create_stemmer()

def preprocess(teks):
    teks = str(teks).lower()
    teks = re.sub(r'[^a-z\s]', ' ', teks)
    teks = re.sub(r'\s+', ' ', teks).strip()
    kata = [w for w in teks.split() if w not in all_stopwords and len(w) > 1]
    return ' '.join([stemmer.stem(w) for w in kata])

print("Menjalankan preprocessing... (1-2 menit)")
df_raw['ulasan_bersih'] = df_raw['ulasan'].apply(preprocess)
print(f"✅ Preprocessing selesai! {len(df_raw)} ulasan diproses.")
print()
print("Contoh hasil preprocessing:")
print("-" * 60)
for _, row in df_raw[['ulasan','ulasan_bersih','Y_Label']].head(5).iterrows():
    print(f"ASLI    : {row['ulasan'][:70]}...")
    print(f"BERSIH  : {row['ulasan_bersih']}")
    print(f"LABEL   : {row['Y_Label']}")
    print()


In [ ]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 3),
    min_df=2,
    sublinear_tf=True
)

X = tfidf.fit_transform(df_raw['ulasan_bersih'])
y = df_raw['Y_Kode'].values

print(f"✅ TF-IDF selesai!")
print(f"   Shape matriks : {X.shape}")
print(f"   Jumlah fitur  : {X.shape[1]} kata/frasa")
print(f"   Jumlah dokumen: {X.shape[0]}")
print()

# Top 20 kata bobot TF-IDF tertinggi
fitur_names = tfidf.get_feature_names_out()
mean_tfidf  = np.asarray(X.mean(axis=0)).flatten()
top20_idx   = mean_tfidf.argsort()[-20:][::-1]

print("Top 20 kata dengan bobot TF-IDF tertinggi:")
for i, idx in enumerate(top20_idx):
    print(f"  {i+1:2d}. {fitur_names[idx]:25s} {mean_tfidf[idx]:.4f}")


In [ ]:
# Visualisasi Top 20 kata TF-IDF
top20_kata  = [fitur_names[i] for i in top20_idx]
top20_skor  = [mean_tfidf[i] for i in top20_idx]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(top20_kata[::-1], top20_skor[::-1], color='#2E75B6')
ax.set_xlabel('Rata-rata Bobot TF-IDF')
ax.set_title('Top 20 Kata dengan Bobot TF-IDF Tertinggi', fontsize=13)
plt.tight_layout()
plt.savefig('top20_tfidf.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik tersimpan sebagai top20_tfidf.png")


## Sel 3c — Fitur Tambahan: Leksikon Sentimen (Deteksi Ulasan 'Campuran')

TF-IDF menganggap tiap kata berdiri sendiri, jadi model kesulitan menangkap pola
**"kata positif DAN negatif muncul bersamaan dalam 1 ulasan"** — padahal itu ciri khas
kelas 'Cukup Puas' ('bagus TAPI kurang nyaman').

Di sini kita pakai **InSet (Indonesia Sentiment Lexicon)** — kamus 3.609 kata positif
dan 6.609 kata negatif (Koto & Rahmaningtyas, IALP 2017) — untuk membuat 3 fitur baru
per ulasan, di luar TF-IDF:
- `n_kata_positif` = jumlah kata dari ulasan yang cocok kamus positif
- `n_kata_negatif` = jumlah kata dari ulasan yang cocok kamus negatif
- `fitur_campuran` = 1 jika ulasan mengandung KEDUA jenis kata (positif & negatif), 0 jika tidak

Fitur `fitur_campuran` inilah yang secara eksplisit menandai ulasan bersentimen ganda —
sesuatu yang tidak bisa ditangkap TF-IDF biasa.

In [ ]:
# Unduh leksikon InSet langsung dari repo resminya
url_pos = 'https://raw.githubusercontent.com/fajri91/InSet/master/positive.tsv'
url_neg = 'https://raw.githubusercontent.com/fajri91/InSet/master/negative.tsv'

lex_pos = pd.read_csv(url_pos, sep='\t')
lex_neg = pd.read_csv(url_neg, sep='\t')

kamus_positif = set(lex_pos['word'].str.lower())
kamus_negatif = set(lex_neg['word'].str.lower())

print(f"✅ Leksikon InSet dimuat: {len(kamus_positif)} kata positif, {len(kamus_negatif)} kata negatif")

def hitung_fitur_leksikon(teks_bersih):
    kata  = str(teks_bersih).split()
    n_pos = sum(1 for k in kata if k in kamus_positif)
    n_neg = sum(1 for k in kata if k in kamus_negatif)
    campuran = 1 if (n_pos > 0 and n_neg > 0) else 0
    return pd.Series([n_pos, n_neg, campuran])

df_raw[['n_kata_positif','n_kata_negatif','fitur_campuran']] = \
    df_raw['ulasan_bersih'].apply(hitung_fitur_leksikon)

print("\nRata-rata fitur per kelas (cek apakah 'fitur_campuran' memang lebih tinggi di 'Cukup Puas'):")
print(df_raw.groupby('Y_Label')[['n_kata_positif','n_kata_negatif','fitur_campuran']].mean().to_string())

# Gabungkan 3 fitur baru ini ke matriks TF-IDF (X)
from scipy.sparse import hstack, csr_matrix

fitur_leksikon = csr_matrix(df_raw[['n_kata_positif','n_kata_negatif','fitur_campuran']].values.astype(float))
X = hstack([X, fitur_leksikon]).tocsr()

# Perbarui daftar nama fitur (dipakai lagi di visualisasi Top 10 kata per kelas)
fitur_names = np.concatenate([fitur_names, ['n_kata_positif','n_kata_negatif','fitur_campuran']])

print(f"\n✅ Fitur leksikon ditambahkan. Total fitur sekarang: {X.shape[1]} (TF-IDF + 3 fitur leksikon)")

## Sel 3b — Tuning Otomatis: Seleksi Fitur Chi-Square + Alpha + Prior

Berdasarkan studi literatur (Peningkatan Performa Naive Bayes dengan Chi-Square:
akurasi 80,66% → 86,22%; Chi-Square + NB untuk sentimen Facebook: hingga 91%),
seleksi fitur **Chi-Square** terbukti secara konsisten meningkatkan akurasi Naive Bayes,
karena membuang kata-kata yang tidak informatif/tidak berhubungan kuat dengan salah satu
kelas, dan hanya menyisakan kata-kata yang paling 'membedakan' antar kelas.

Di sini kita cari otomatis: jumlah fitur terbaik (`k`) hasil Chi-Square,
dikombinasikan dengan `alpha` dan `class_prior` terbaik, sekaligus.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Pipeline: seleksi fitur Chi-Square -> Multinomial Naive Bayes
pipe_cs = Pipeline([
    ('select', SelectKBest(score_func=chi2)),
    ('nb', MultinomialNB())
])

jumlah_fitur_tersedia = X.shape[1]
print(f"Total fitur TF-IDF tersedia: {jumlah_fitur_tersedia}")

# Urutan class_prior mengikuti LABELS_ORDER = [0,1,2] = [Tidak Puas, Cukup Puas, Puas]
param_grid = {
    'select__k': sorted(set([
        int(jumlah_fitur_tersedia*0.10), int(jumlah_fitur_tersedia*0.25),
        int(jumlah_fitur_tersedia*0.50), int(jumlah_fitur_tersedia*0.75),
        jumlah_fitur_tersedia
    ])),
    'nb__alpha': [0.1, 0.5, 1.0, 2.0],
    'nb__class_prior': [
        None,                    # ikuti proporsi data asli (paling condong ke kelas mayoritas)
        [0.375, 0.25, 0.375],    # bantu 'Cukup Puas' sedikit
        [0.35, 0.30, 0.35],      # bantu 'Cukup Puas' sedang
        [1/3, 1/3, 1/3],         # seragam penuh (dipakai di percobaan sebelumnya)
        [0.30, 0.40, 0.30],      # bantu 'Cukup Puas' lebih besar dari proporsi seragam
    ]
}

grid = GridSearchCV(
    estimator=pipe_cs,
    param_grid=param_grid,
    cv=skf,
    scoring='accuracy',
    n_jobs=-1
)
print("Menjalankan Grid Search (Chi-Square k x alpha x class_prior)... ini bisa beberapa menit\n")
grid.fit(X, y)

hasil_tuning = pd.DataFrame(grid.cv_results_)[
    ['param_select__k','param_nb__alpha','param_nb__class_prior','mean_test_score']
].sort_values('mean_test_score', ascending=False).reset_index(drop=True)

print("=== TOP 10 HASIL TUNING (diurutkan dari akurasi CV tertinggi) ===")
print(hasil_tuning.head(10).to_string(index=True))

k_terbaik     = grid.best_params_['select__k']
alpha_terbaik = grid.best_params_['nb__alpha']
prior_terbaik = grid.best_params_['nb__class_prior']

print(f"\n>>> Kombinasi TERBAIK: k={k_terbaik} fitur, alpha={alpha_terbaik}, class_prior={prior_terbaik}")
print(f">>> Akurasi CV terbaik: {grid.best_score_*100:.2f}%")

## Sel 3d — Bandingkan dengan ComplementNB

`ComplementNB` adalah varian Naive Bayes yang secara khusus dirancang scikit-learn
untuk data teks yang **tidak seimbang antar kelas** (persis situasi kita: Tidak Puas=151,
Cukup Puas=77, Puas=165). Caranya berbeda dari MultinomialNB: alih-alih mempelajari
"ciri khas kata di kelas ini", ia mempelajari "ciri khas kata di SEMUA kelas LAIN",
yang secara matematis terbukti lebih stabil untuk kelas minoritas.

Di sini kita bandingkan langsung head-to-head dengan MultinomialNB (hasil Sel 3b),
lalu otomatis pakai yang akurasinya lebih tinggi.

In [ ]:
from sklearn.naive_bayes import ComplementNB

pipe_cnb = Pipeline([
    ('select', SelectKBest(score_func=chi2)),
    ('cnb', ComplementNB())
])

param_grid_cnb = {
    'select__k': param_grid['select__k'],
    'cnb__alpha': [0.1, 0.5, 1.0, 2.0],
    'cnb__norm': [True, False]
}

grid_cnb = GridSearchCV(
    estimator=pipe_cnb,
    param_grid=param_grid_cnb,
    cv=skf,
    scoring='accuracy',
    n_jobs=-1
)
print("Menjalankan Grid Search untuk ComplementNB...\n")
grid_cnb.fit(X, y)

print(f"Akurasi CV terbaik MultinomialNB : {grid.best_score_*100:.2f}%")
print(f"Akurasi CV terbaik ComplementNB  : {grid_cnb.best_score_*100:.2f}%")

if grid_cnb.best_score_ > grid.best_score_:
    print("\n>>> ComplementNB LEBIH BAIK — dipakai untuk pemodelan final di Sel 4.")
    algoritma_terbaik = 'complement'
    k_terbaik     = grid_cnb.best_params_['select__k']
    alpha_terbaik = grid_cnb.best_params_['cnb__alpha']
    norm_terbaik  = grid_cnb.best_params_['cnb__norm']
else:
    print("\n>>> MultinomialNB tetap sama/lebih baik — tetap dipakai untuk pemodelan final di Sel 4.")
    algoritma_terbaik = 'multinomial'

## Sel 4 — Modeling: Multinomial Naïve Bayes (CRISP-DM Tahap 4)

**Rumus dasar Naïve Bayes:**

$$P(kelas | teks) = \frac{P(teks | kelas) \cdot P(kelas)}{P(teks)}$$

**Mengapa Multinomial NB?**
Data TF-IDF menghasilkan nilai frekuensi kata (non-negatif),
sehingga Multinomial NB lebih tepat dibanding Gaussian NB.

**Parameter:**
- `k`, `alpha`, & `class_prior` = hasil tuning otomatis di Sel 3b (Chi-Square + Grid Search)
- `n_splits=10` = 10-Fold Stratified Cross Validation


In [ ]:
# Pipeline final: seleksi fitur Chi-Square (k_terbaik) + algoritma NB terbaik hasil Sel 3d
# Semua parameter hasil tuning otomatis, bukan ditebak manual
if algoritma_terbaik == 'complement':
    pipeline_final = Pipeline([
        ('select', SelectKBest(score_func=chi2, k=k_terbaik)),
        ('nb', ComplementNB(alpha=alpha_terbaik, norm=norm_terbaik))
    ])
    print(f"Memakai ComplementNB (alpha={alpha_terbaik}, norm={norm_terbaik}, k={k_terbaik})")
else:
    pipeline_final = Pipeline([
        ('select', SelectKBest(score_func=chi2, k=k_terbaik)),
        ('nb', MultinomialNB(alpha=alpha_terbaik, class_prior=prior_terbaik))
    ])
    print(f"Memakai MultinomialNB (alpha={alpha_terbaik}, class_prior={prior_terbaik}, k={k_terbaik})")

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

print("Menjalankan 10-Fold Cross Validation...")

# Prediksi cross-validation (seleksi fitur otomatis di-fit ulang di tiap fold latih,
# supaya TIDAK ada kebocoran data/data leakage dari data uji)
y_pred  = cross_val_predict(pipeline_final, X, y, cv=skf)
y_proba = cross_val_predict(pipeline_final, X, y, cv=skf, method='predict_proba')

# Latih pipeline final dengan seluruh data
pipeline_final.fit(X, y)

# Ekstrak model & selector supaya sel-sel berikutnya (Top 10 kata, prediksi ulasan baru)
# tetap bisa dipakai tanpa perlu diubah strukturnya
model    = pipeline_final.named_steps['nb']
selector = pipeline_final.named_steps['select']

print(f"✅ Pemodelan selesai!")
print(f"   Total data diprediksi : {len(y_pred)}")
print(f"   Distribusi prediksi   :")
LABEL_MAP  = {0:'Tidak Puas', 1:'Cukup Puas', 2:'Puas'}
unique, cnt = np.unique(y_pred, return_counts=True)
for u, c in zip(unique, cnt):
    print(f"     {LABEL_MAP[u]:12s}: {c}")


## Sel 5 — Evaluasi Model (CRISP-DM Tahap 5)

In [ ]:
LABELS_TEXT  = ['Tidak Puas', 'Cukup Puas', 'Puas']
LABELS_ORDER = [0, 1, 2]

# Akurasi
acc = accuracy_score(y, y_pred)
print(f"Akurasi Keseluruhan: {acc*100:.2f}%\n")

# Confusion Matrix
cm    = confusion_matrix(y, y_pred, labels=LABELS_ORDER)
cm_df = pd.DataFrame(cm,
          index   =[f'True: {l}' for l in LABELS_TEXT],
          columns =[f'Pred: {l}' for l in LABELS_TEXT])
print("Confusion Matrix:")
print(cm_df)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABELS_TEXT, yticklabels=LABELS_TEXT,
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Confusion Matrix — Multinomial Naïve Bayes', fontsize=13)
axes[0].set_xlabel('Prediksi'); axes[0].set_ylabel('Aktual')

# ROC Curve
y_bin         = label_binarize(y, classes=LABELS_ORDER)
auc_per_class = roc_auc_score(y_bin, y_proba, multi_class='ovr', average=None)
auc_macro     = roc_auc_score(y_bin, y_proba, multi_class='ovr', average='macro')
colors        = ['#E74C3C', '#F39C12', '#27AE60']

for i, (lbl, color) in enumerate(zip(LABELS_TEXT, colors)):
    fpr, tpr, _ = roc_curve(y_bin[:,i], y_proba[:,i])
    axes[1].plot(fpr, tpr, color=color, lw=2,
                 label=f'{lbl} (AUC={auc_per_class[i]:.3f})')
axes[1].plot([0,1],[0,1],'--', color='gray', label='Random Guess')
axes[1].set_title('ROC Curve — One-vs-Rest (3 Kelas)', fontsize=13)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig('evaluasi_model.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik tersimpan sebagai evaluasi_model.png")


In [ ]:
# Classification Report lengkap
report = classification_report(y, y_pred, target_names=LABELS_TEXT, output_dict=True)
print(classification_report(y, y_pred, target_names=LABELS_TEXT))

# Tabel ringkasan untuk Bab IV
ringkasan = pd.DataFrame({
    'Kelas'      : LABELS_TEXT,
    'Precision'  : [report[l]['precision'] for l in LABELS_TEXT],
    'Recall'     : [report[l]['recall']    for l in LABELS_TEXT],
    'F1-Score'   : [report[l]['f1-score']  for l in LABELS_TEXT],
    'AUC (OvR)'  : auc_per_class,
    'Support'    : [int(report[l]['support']) for l in LABELS_TEXT],
})

print(f"Akurasi Keseluruhan : {acc*100:.2f}%")
print(f"AUC Macro Average   : {auc_macro:.3f}\n")
display(ringkasan.round(4))


In [ ]:
# Visualisasi Top 10 Kata per Kelas
# Catatan: 'model' sekarang dilatih di atas FITUR HASIL SELEKSI CHI-SQUARE (bukan semua fitur
# tfidf), jadi nama fiturnya juga harus disaring pakai selector supaya indeksnya tetap cocok.
fitur_names_terpilih = fitur_names[selector.get_support()]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#E74C3C', '#F39C12', '#27AE60']

for cls_idx, (cls_name, color) in enumerate(zip(LABELS_TEXT, colors)):
    log_prob  = model.feature_log_prob_[cls_idx]
    top10_idx = log_prob.argsort()[-10:][::-1]
    top10_kata = [fitur_names_terpilih[i] for i in top10_idx]
    top10_skor = [log_prob[i] for i in top10_idx]

    axes[cls_idx].barh(top10_kata[::-1], top10_skor[::-1], color=color)
    axes[cls_idx].set_title(f'Top 10 Kata\n{cls_name}', fontsize=12)
    axes[cls_idx].set_xlabel('Log Probability')

plt.suptitle('Kata Paling Berpengaruh per Kelas Kepuasan', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('kata_per_kelas.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik tersimpan sebagai kata_per_kelas.png")


## Sel 6 — Simpan Model ke File (untuk Dashboard Streamlit Cloud)

Model dan TF-IDF vectorizer disimpan ke file `.pkl` agar dashboard
Streamlit Cloud bisa langsung memuatnya **tanpa perlu melatih ulang**
setiap kali halaman dibuka.


In [ ]:
# Simpan model dan vectorizer
joblib.dump(model, 'model_geoffmax.pkl')
joblib.dump(tfidf, 'tfidf_geoffmax.pkl')

print("✅ Model berhasil disimpan!")
print("   model_geoffmax.pkl  — model Naive Bayes terlatih")
print("   tfidf_geoffmax.pkl  — TF-IDF vectorizer")
print()
print("Langkah selanjutnya:")
print("1. Download kedua file .pkl ini ke komputer Anda")
print("2. Upload ke GitHub bersama app_geoffmax.py dan requirements.txt")
print("3. Deploy ke share.streamlit.io")

# Auto-download kedua file
from google.colab import files
files.download('model_geoffmax.pkl')
files.download('tfidf_geoffmax.pkl')


## Sel 7 — Demo Prediksi Ulasan Baru

Uji coba model dengan memasukkan ulasan baru secara manual.


In [ ]:
def prediksi_ulasan(teks_baru):
    teks_bersih  = preprocess(teks_baru)
    tfidf_input  = tfidf.transform([teks_bersih])
    # Hitung fitur leksikon yang sama seperti saat training
    n_pos, n_neg, campuran = hitung_fitur_leksikon(teks_bersih)
    fitur_tambahan_input = csr_matrix([[n_pos, n_neg, campuran]], dtype=float)
    tfidf_input  = hstack([tfidf_input, fitur_tambahan_input]).tocsr()
    tfidf_input  = selector.transform(tfidf_input)  # samakan dengan fitur hasil Chi-Square
    pred_kelas   = model.predict(tfidf_input)[0]
    pred_proba   = model.predict_proba(tfidf_input)[0]

    emoji = {0:'😞', 1:'😐', 2:'😊'}
    print(f"Ulasan   : {teks_baru}")
    print(f"Bersih   : {teks_bersih}")
    print(f"Prediksi : {emoji[pred_kelas]} {LABEL_MAP[pred_kelas]}")
    print("Probabilitas:")
    for lbl, prob in zip(LABELS_TEXT, pred_proba):
        bar = '█' * int(prob * 30)
        print(f"  {lbl:12s}: {bar} {prob*100:.1f}%")
    print()

# Uji dengan 3 contoh ulasan
prediksi_ulasan("sepatunya bagus banget kualitas oke nyaman dipakai seharian")
prediksi_ulasan("kecewa banget kualitas buruk sol lepas setelah 3 hari dipakai")
prediksi_ulasan("lumayan sih untuk harga segini tidak jelek tidak bagus juga")
